# Building out Makemore MLP
### This time around, we give in input 3 characters, and expect a fourth char to be generated

In [23]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [24]:
words = open('names.txt', 'r').read().splitlines()

In [25]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [26]:
# now, we need to construct the dataset.
block_size = 3
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

# now, we want to divide this into test, train and val sets.
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xval, Yval = build_dataset(words[n1:n2])
Xtest, Ytest = build_dataset(words[n2:])


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [ ]:
# building out train, test, and val datasets.

X, Y = build_dataset(words)

In [27]:
# now, we have to embed each character as an x-dimensional tensor.
# what even is an embedding? it's a random vector that we initialize for each character. initially, it contains nonsense values.
# but, as we train the model, the embeddings will be updated, and they will start to actually represent the chars meaningfully.
# why are we doing this instead of just one-hot encoding the chars?
# because, when we have chars in latent space embedded close to each other, we know that they are somewhat semantically related (in some words atleast), and they can be the catalyst for the model to generate unique and new results.

# ok, so, how are we going to implement this?
# we will have C, an embedding tensor that contains the embeddings of all 3 characters being given in input.

C = torch.randn((27, 2), requires_grad=True)
W1 = torch.randn(6, 100, requires_grad=True) # hidden layer weights.
b1 = torch.randn(100, requires_grad=True)
W2 = torch.randn(100, 27, requires_grad=True)
b2 = torch.randn(27, requires_grad=True)

params = [C, W1, b1, W2, b2]

In [ ]:
# so, what is C[X] actually doing?
# so, it takes C, which is a (27, 2) tensor, and X, 

emb = C[X]
emb.shape

In [ ]:
# now, each chars embedding is supposed to be (1, 3). And, each tri-gram's embedding is a concatenation of it's constituent 3 char's embeddings.
# so, each trigram's embedding is of shape (1, 3, 2).
# and, we have 32 trigrams in a single batch, i.e. every 32 tri-grams, our network updates it's parameters.
# so, when we concatenate 32 trigrams, we get a tensor of shape (32, 3, 2).

# this is all well and good, but, this embeddings of (32, 3, 2) can't be fed into the hidden layer that's supposed to come next. Because it's a 3-dim tensor, and by convention, neural net layers expect a 2-d tensor.
# so, we flatten (32, 3, 2) into (32, 6), and then we can feed it into the hidden layer.
# note: i'm saying 32 in (32, ...) shaped tensors here, because 32 is going to be my batch size, but realistically, it could be any size.

emb_flat = emb.view(-1, 6)
emb_flat.shape

In [ ]:
# now, we pass in the flattened tensor through the weight matrix of the hidden layer.
# how are we going to initialize it tho? what dims should it be?
# let's see. first, we have the input embeddings tensor, which is going to be of shape (32, 6), so, the hidden layer should be of dim (6, ..)
# the second dimension, i.e. the number of neurons, is up to us. It can be 100, or 300, or 1000, etc. let's stick to 100, for now.

activ = torch.tanh(emb_flat @ W1 + b1)
activ.shape

In [ ]:
# now that we have the activations of the hidden layer, we want to now, pass it through the last layer, i.e. the linear layer.
# since the activations are of the shape (32, 100), the linear layer is going to have to be of the shape (100, 27), becuase, one: we match end dims.
# and two, we want our output to basically be probabilities of all characters. So, we want a (32, 27) shaped output, which is nothing but logits.
# out of the (32, 27), i.e 32 rows, each row corresponds to each input. (remember, we had 32 tri-grams as inputs.)

logits = activ @ W2 + b2
logits.shape


In [ ]:
# now, to calculate the loss.
# we want to calculate either of the -ve log likelihood loss or the cross entropy loss.
# so, first, we have to calculate the probs, which is done using softmax.

exps = torch.exp(logits)
probs = exps / torch.sum(exps, dim=1, keepdim=True)

In [ ]:
loss = F.cross_entropy(logits, Y)
loss

In [ ]:
# now, for the backward pass
for p in params:
    p.grad = None

loss.backward() # fills gradients

In [ ]:
# now, time to update the weights and biases using the gradients that we just calculated.
# we have to update W2, b2, W1, b1 and C
# all of these are our parameters

for p in params:
    p.data += -0.1 * p.grad

In [28]:
# now, for the training loop.

for i in range(10000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    minibatch, minitarget = Xtr[ix], Ytr[ix]
    emb = C[minibatch]
    emb_flat = emb.view(-1, 6)
    activ = torch.tanh(emb_flat @ W1 + b1) # hidden layer
    logits = activ @ W2 + b2 # linear layer

    loss = F.cross_entropy(logits, minitarget) # loss
    print(i, loss.item())

    for p in params:
        p.grad = None

    loss.backward() # fills gradients

    # if i > 6000:
    #     lr = 0.001
    # else:
    #     lr = 0.1

    lr = 0.2

    # update
    for p in params:
        p.data += -lr * p.grad

0 13.364847183227539
1 13.116365432739258
2 11.338644027709961
3 10.601016998291016
4 8.839338302612305
5 11.133389472961426
6 8.463140487670898
7 9.325645446777344
8 7.9297075271606445
9 7.302244186401367
10 8.398825645446777
11 7.4761857986450195
12 7.9347639083862305
13 7.067924499511719
14 5.852385520935059
15 7.200096130371094
16 7.261031150817871
17 5.30570125579834
18 7.846221446990967
19 4.4301557540893555
20 6.2082319259643555
21 5.370113849639893
22 6.12929105758667
23 5.54612922668457
24 5.608206748962402
25 3.8338069915771484
26 6.157374858856201
27 5.456172466278076
28 4.739209175109863
29 4.768886089324951
30 4.219135284423828
31 6.001397609710693
32 5.879382610321045
33 5.018160820007324
34 5.259067058563232
35 5.200030326843262
36 4.85363245010376
37 4.391387462615967
38 4.010496616363525
39 4.286466598510742
40 4.4394307136535645
41 4.722435474395752
42 4.0613789558410645
43 5.496902942657471
44 4.92420768737793
45 3.181614875793457
46 4.248718738555908
47 2.8826150894

In [30]:
# now, let's sample from the model.

g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

cramah.
ame.
hler.
kemli.
rev.
cren.
aden.
jrahnen.
dellyst.
vaeri.
jer.
ara.
ceriiv.
malec.
phh.
maizi.
derinn.
sroi.
rar.
vabi.
